# Transformacion y preparacion de datos

En este notebook se prepara una copia del catalogo de USGS para los analisis posteriores. El archivo de `data/raw/` se conserva sin modificaciones: las transformaciones se realizan sobre `df` en memoria y al final se guarda una copia en `data/processed/`.

# 1. Importaciones y rutas


In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import Polygon


def find_project_root():
    """Busca la carpeta que contiene el CSV crudo."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "sismos_usgs.csv").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto")


PROJECT_ROOT = find_project_root()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "sismos_usgs.csv"
GEO_PATH = PROJECT_ROOT / "data" / "raw" / "geodata"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH = PROCESSED_DIR / "sismos_argentina_limpio.csv"

# 2. Carga y copia de trabajo

Primero se carga el archivo original y se crea una copia. De esta forma, `df_raw` sirve como referencia y todas las transformaciones se realizan sobre `df`.

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()

required_columns = {
    "id", "time", "updated", "latitude", "longitude", "depth", "mag",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {sorted(missing_columns)}")

print(f"Filas iniciales: {df.shape[0]:,}")
print(f"Columnas iniciales: {df.shape[1]}")
print(f"El archivo crudo conserva {df_raw.shape[0]:,} filas")

# 3. Revision inicial de calidad

Antes de eliminar o imputar datos se observa la cantidad de nulos, duplicados y los tipos de cada columna.

In [ ]:
print("Tipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum().sort_values(ascending=False))

print(f"\nDuplicados completos: {df.duplicated().sum()}")
print(f"IDs duplicados: {df['id'].duplicated().sum()}")

# 4. Seleccion geografica de Argentina

Se repite el criterio espacial del notebook 01: un evento pertenece al area de estudio si su punto queda dentro de las provincias, la plataforma continental o el poligono antartico utilizado por el proyecto. El filtro se realiza sobre una copia y no cambia el CSV original.

In [ ]:
gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",
)

provincias = gpd.read_file(GEO_PATH / "provinciaPolygon.shp")
plataforma = gpd.read_file(GEO_PATH / "plataforma_continentalPolygon.shp")

coordenadas_antartida = [
    (-74.0, -60.0), (-25.0, -60.0), (-25.0, -90.0),
    (-74.0, -90.0), (-74.0, -60.0),
]
poligono_antartico = Polygon(coordenadas_antartida)
poligono_nacional = (
    provincias.geometry.union_all()
    .union(plataforma.geometry.union_all())
    .union(poligono_antartico)
)

df = pd.DataFrame(
    gdf[gdf.geometry.within(poligono_nacional)].drop(columns="geometry")
)
print(f"Filas dentro del area de estudio: {df.shape[0]:,}")

# 5. Eliminación de columnas

En el EDA se identificaron seis columnas que no aportan información útil para el análisis. Se eliminan por dos motivos distintos.

**Por exceso de valores faltantes:**

- `nst` (cantidad de estaciones sísmicas usadas para localizar el evento): le falta el valor en el 60,2 % de los registros (3.873 de 6.429). Con tantos faltantes, imputarla implicaría inventar la mayor parte de la columna, y eliminar esas filas implicaría perder más de la mitad de los datos.

**Por falta de variabilidad:** en estas columnas una sola categoría concentra casi todos los registros. Una variable que tiene prácticamente el mismo valor en todas las filas no ayuda a distinguir un sismo de otro.

- `net` (red sísmica que aportó la solución preferida): 6.427 de 6.429 registros son `us`. Los dos restantes son los registros `iscgem` que se eliminan en la sección siguiente.
- `type` (tipo de evento): el 100 % de los registros son `earthquake`, así que no hay explosiones ni otros tipos de evento.
- `status` (estado de revisión): el 100 % está `reviewed`, es decir, todos los eventos fueron revisados por un sismólogo.
- `locationSource` (red que calculó la ubicación): el 99,9 % es `us`.
- `magSource` (red que calculó la magnitud): el 96,5 % es `us`.

La columna `magType` también es categórica, pero se conserva: presenta varias escalas de magnitud con cantidades relevantes (`mb`, `mww`, `mwr`, `ml`), así que sí aporta información.

In [ ]:
columns_to_drop = [
    "nst", "net", "type", "status", "locationSource", "magSource",
]
existing_columns_to_drop = [
    column for column in columns_to_drop if column in df.columns
]

columnas_antes = df.shape[1]
df = df.drop(columns=existing_columns_to_drop)

pd.DataFrame({
    "Antes": [columnas_antes],
    "Después": [df.shape[1]],
    "Columnas eliminadas": [", ".join(existing_columns_to_drop)],
}, index=["Cantidad de columnas"])

# 6. Eliminación de registros incompletos

En el EDA se identificaron cuatro eventos a los que les faltan casi todas las variables de calidad (`nst`, `gap`, `dmin`, `rms`, `horizontalError`, `magNst`). Estas variables indican qué tan confiable es la medición, así que sin ellas no hay forma de evaluar si el dato es correcto.

- `iscgem621613005` e `iscgem620210242`: además de los faltantes, tienen magnitudes atípicas (5,06 y 5,18) respecto del resto del catálogo, y provienen de una red distinta (`iscgem`) a la del resto de los datos.
- `us10007f0t` y `us100073ln`: tienen 5 o más variables de calidad faltantes.

El resto de los registros con faltantes tiene como máximo 3 valores nulos y se conserva, porque se puede completar sin perder confiabilidad (ver sección 8).

Los registros se eliminan por su `id`, que es un identificador estable, y no por su número de fila, que cambia cada vez que se filtran los datos.

In [ ]:
suspicious_ids = {"iscgem621613005", "iscgem620210242"}
present_suspicious_ids = set(df.loc[df["id"].isin(suspicious_ids), "id"])

registros_eliminados = df[df["id"].isin(present_suspicious_ids)]
filas_antes = len(df)

df = df.loc[~df["id"].isin(present_suspicious_ids)].copy()

display(registros_eliminados)

pd.DataFrame(
    {"Filas": [filas_antes, len(df)]},
    index=["Antes", "Después"],
)

# 7. Fechas y variables temporales

Las columnas `time` (momento en que ocurrió el sismo) y `updated` (última actualización del registro) vienen como texto. Mientras sean texto no se pueden ordenar cronológicamente, calcular diferencias de tiempo ni agrupar por período. Por eso se convierten a tipo fecha, en horario UTC, que es el que usa USGS.

A partir de `time` se crean cuatro variables nuevas:

- `event_year`: año del evento. Permite analizar la evolución de la actividad sísmica a lo largo de los años.
- `event_month`: mes del evento. Permite detectar posibles patrones estacionales.
- `event_day`: día del mes.
- `event_hour`: hora del evento (UTC).

Estas variables se van a usar en el análisis temporal y en el modelo supervisado, que necesita separar los datos por períodos.

In [ ]:
for column in ["time", "updated"]:
    df[column] = pd.to_datetime(df[column], utc=True, errors="raise")

df["event_year"] = df["time"].dt.year
df["event_month"] = df["time"].dt.month
df["event_day"] = df["time"].dt.day
df["event_hour"] = df["time"].dt.hour

print(df[["time", "updated", "event_year", "event_month", "event_day", "event_hour"]].head())

# 8. Tratamiento de valores faltantes

Primero se verifica que ningún registro tenga faltantes en las variables principales (`latitude`, `longitude`, `depth` y `mag`). Sin ubicación, profundidad o magnitud, un sismo no se puede analizar, así que esas filas se eliminarían en lugar de completarse. En este catálogo no hay ninguna.

Después se completan los faltantes de las variables de calidad que quedan. En todas representan una proporción baja del total:

| Variable | Qué mide | % faltante |
|---|---|---|
| `gap` | Brecha azimutal entre estaciones (grados) | 0,12 % |
| `dmin` | Distancia a la estación más cercana (grados) | 0,12 % |
| `rms` | Error de ajuste de los tiempos de llegada (segundos) | 0,03 % |
| `horizontalError` | Incertidumbre de la ubicación horizontal (km) | 0,03 % |
| `magError` | Incertidumbre de la magnitud | 3,5 % |
| `magNst` | Estaciones usadas para calcular la magnitud | 3,6 % |

Se usa la **mediana** y no la media porque estas variables tienen valores extremos (como se vio en los boxplots del EDA). La media se desplaza hacia esos valores, mientras que la mediana representa mejor el valor típico.

`depthError` no necesita imputación porque no tiene faltantes, y `nst` ya fue eliminada en la sección 5.

In [ ]:
essential_columns = ["latitude", "longitude", "depth", "mag"]
rows_before_missing = len(df)
df = df.dropna(subset=essential_columns).copy()

numeric_imputation_columns = [
    "gap", "dmin", "rms", "horizontalError", "magError", "magNst",
]
numeric_imputation_columns = [
    column for column in numeric_imputation_columns if column in df.columns
]

nulos_antes = df[numeric_imputation_columns].isnull().sum()
medianas = {}

for column in numeric_imputation_columns:
    df[column] = pd.to_numeric(df[column], errors="raise")
    median = df[column].median()
    if pd.isna(median):
        raise ValueError(f"No se puede imputar {column}: mediana nula")
    medianas[column] = median
    df[column] = df[column].fillna(median)

display(pd.DataFrame(
    {"Filas": [rows_before_missing, len(df)]},
    index=["Antes de eliminar faltantes esenciales", "Después"],
))

pd.DataFrame({
    "Nulos antes": nulos_antes,
    "Mediana utilizada": pd.Series(medianas).round(3),
    "Nulos después": df[numeric_imputation_columns].isnull().sum(),
})

# 9. Comprobaciones finales

Se verifica automáticamente que el resultado cumpla con lo esperado:

- cada `id` aparece una sola vez y no hay filas duplicadas;
- `time` y `updated` son de tipo fecha;
- no quedan faltantes en las variables principales ni en las imputadas.

Si alguna condición no se cumple, el notebook se detiene con un error. Después se compara el catálogo original con el resultado final.

In [ ]:
assert not df["id"].duplicated().any()
assert not df.duplicated().any()
assert isinstance(df["time"].dtype, pd.DatetimeTZDtype)
assert isinstance(df["updated"].dtype, pd.DatetimeTZDtype)
assert df[essential_columns].notna().all().all()
assert df[numeric_imputation_columns].notna().all().all()

resumen = pd.DataFrame({
    "Antes (archivo crudo)": [
        df_raw.shape[0], df_raw.shape[1],
        df_raw.duplicated().sum(), int(df_raw.isna().sum().sum()),
    ],
    "Después (transformado)": [
        df.shape[0], df.shape[1],
        df.duplicated().sum(), int(df.isna().sum().sum()),
    ],
}, index=["Filas", "Columnas", "Filas duplicadas", "Valores nulos"])

display(resumen)
display(df.describe(include="all").transpose())

# 10. Visualizaciones de control

Como última revisión se grafican dos cosas:

- La distribución de magnitudes, para confirmar que no quedaron valores atípicos después de eliminar los registros incompletos.
- La relación entre magnitud y profundidad, para detectar combinaciones que no tengan sentido físico.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["mag"], bins=20, edgecolor="black")
axes[0].set_title("Distribucion de magnitud")
axes[0].set_xlabel("Magnitud")
axes[0].set_ylabel("Cantidad de eventos")

axes[1].scatter(df["depth"], df["mag"], alpha=0.5)
axes[1].set_title("Magnitud y profundidad")
axes[1].set_xlabel("Profundidad")
axes[1].set_ylabel("Magnitud")

plt.tight_layout()
plt.show()

# 11. Guardado del dataset procesado

Se guarda una copia procesada en `data/processed/` para que las siguientes etapas puedan reutilizarla sin ejecutar nuevamente toda la transformacion. El archivo crudo de `data/raw/` no se modifica.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)

## Conclusion

El DataFrame `df` y el archivo procesado quedan listos para continuar con la etapa de visualizacion.